In [ ]:
import io, sys
import math
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

sys.path.append(rf"/home/susanket/esri-vlm/src")
sys.path.append(rf"/home/susanket/esri-vlm")

# Imports from eval.py / model codebase
from model import CLIP, build_model
from main import CLIPLightningModule
from sft.loader import load_pretrained_clip
from eval import is_sft_checkpoint, load_sft_clip, IMAGENET_MEAN, IMAGENET_STD

def load_vlm_model(ckpt_path: str, device: torch.device):
    """Loads and initializes the CLIP model and tokenizer from a checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    if is_sft_checkpoint(ckpt):
        model, tokenizer = load_sft_clip(ckpt, device)
    elif "hyper_parameters" in ckpt:
        for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
            ckpt["hyper_parameters"].pop(hp_key, None)
        model_hp = dict(ckpt["hyper_parameters"])
        model_hp["load_text_backbone"] = True
        lightning_model = CLIPLightningModule(**model_hp).to(device)
        lightning_model.load_state_dict(ckpt["state_dict"])
        lightning_model.eval()
        model = lightning_model.model
        tokenizer = lightning_model.tokenizer
    else:
        from transformers import AutoTokenizer
        model = build_model(ckpt["state_dict"]).to(device)
        tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

    model.eval()
    return model, tokenizer


def get_image_transform(image_size: int = 224):
    """Standard evaluation image normalization pipeline."""
    return transforms.Compose([
        transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def search_and_visualize_collage(
    image_parquet_path: str,
    caption_parquet_path: str,
    ckpt_path: str,
    search_phrase: str,
    num_samples: int = 9,
    device_str: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    device = torch.device(device_str)
    
    # 1. Load and merge Parquets
    print("Loading Parquet datasets...")
    df_img = pd.read_parquet(image_parquet_path)
    df_cap = pd.read_parquet(caption_parquet_path)
    df = pd.merge(df_img, df_cap, on="file_name")

    # 2. Text Search across Caption Columns
    search_lower = search_phrase.lower()
    mask = (
        df["key_elements"].astype(str).str.lower().str.contains(search_lower) |
        df["summary"].astype(str).str.lower().str.contains(search_lower) |
        df["dense_caption"].astype(str).str.lower().str.contains(search_lower)
    )
    matched_df = df[mask].copy()

    if len(matched_df) == 0:
        print(f"No caption matches found for '{search_phrase}'. Falling back to full dataset.")
        matched_df = df.copy()

    # 3. Stratified Sampling across diverse Zoom levels
    distinct_zooms = matched_df["zoom"].unique()
    np.random.shuffle(distinct_zooms)
    
    sampled_rows = []
    # Round-robin sampling from distinct zoom levels
    while len(sampled_rows) < num_samples and len(matched_df) > len(sampled_rows):
        for z in distinct_zooms:
            z_group = matched_df[(matched_df["zoom"] == z) & (~matched_df.index.isin([r.name for r in sampled_rows]))]
            if not z_group.empty:
                sampled_rows.append(z_group.sample(1).iloc[0])
                if len(sampled_rows) == num_samples:
                    break

    sample_df = pd.DataFrame(sampled_rows)

    # 4. Load VLM Model and Tokenizer
    print("Loading VLM Model...")
    model, tokenizer = load_vlm_model(ckpt_path, device)
    transform = get_image_transform()

    # 5. Compute Text Embedding
    with torch.no_grad():
        enc = tokenizer(
            [search_phrase],
            padding=True,
            truncation=True,
            max_length=model.transformer.context_length,
            return_tensors="pt"
        )
        tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        text_emb = F.normalize(model.encode_text(tokens), dim=-1)

    # 6. Compute Image Embeddings & Similarities
    pil_images = []
    image_tensors = []

    for _, row in sample_df.iterrows():
        img_bytes = row["image_bytes"]
        try:
            pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            pil_img = Image.new("RGB", (224, 224))
        
        pil_images.append(pil_img)
        image_tensors.append(transform(pil_img))

    image_batch = torch.stack(image_tensors).to(device)

    with torch.no_grad():
        # Handle optional DINO backbone if specified by model config
        if getattr(model.visual, "precomputed", False):
            from model import DINOV2_REPO
            dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name).eval().to(device)
            feats = dino_backbone.forward_features(image_batch)
            image_batch = torch.cat([feats["x_norm_clstoken"].unsqueeze(1), feats["x_norm_patchtokens"]], dim=1)

        img_emb = F.normalize(model.encode_image(image_batch), dim=-1)
        sim_scores = (text_emb @ img_emb.T).squeeze(0).cpu().numpy()

    # 7. Plot 3x3 Collage
    grid_size = math.ceil(math.sqrt(num_samples))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(12, 12))
    fig.suptitle(f"Search Query: '{search_phrase}'", fontsize=16, fontweight="bold")

    axes = axes.flatten()
    for idx in range(num_samples):
        ax = axes[idx]
        ax.imshow(pil_images[idx])
        zoom_val = sample_df.iloc[idx]["zoom"]
        sim_val = sim_scores[idx]
        
        # ax.set_title(f"Zoom: {zoom_val} | Sim: {sim_val:.4f}", fontsize=11, fontweight="semibold")
        ax.set_title(f"{sim_val:.4f}", fontsize=11, fontweight="semibold")
        ax.axis("off")

    # Turn off any unused grid axes
    for idx in range(num_samples, len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
search_and_visualize_collage(
    image_parquet_path="/data/susanket/vlm/images3/image-01.parquet",
    caption_parquet_path="/data/susanket/vlm/images3/caption-01.parquet",
    ckpt_path="/home/susanket/esri-vlm/runs/sft/esri-vlm-sft/z7kv8tpg/checkpoints/epoch=49-step=16950.ckpt",
    search_phrase="swimming pool"
)